### There are several strategy to merge them. Pick your strategy related cells to run

In [ ]:
# —— 配置区域：请修改为你的实际路径 ——
A_PATH = ""   # A 文件路径（is_bug -> "yes"）
B_PATH = ""   # B 文件路径（is_bug -> "no"）
C_PATH = ""

OUTPUT_DIR = ""   # 输出目录

# 随机种子（用于 shuffle 复现）
RANDOM_SEED = 42
NORMALIZE = False

In [6]:
import json
import os
import random
import hashlib
from typing import List, Dict, Any
from collections import Counter

os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_json_list(path: str) -> List[Dict[str, Any]]:
    """要求是 JSON 数组(list) 文件。"""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        if not isinstance(data, list):
            raise ValueError(f"{path} 必须是 JSON 数组！")
        return [x for x in data if isinstance(x, dict)]

def save_as_json(path: str, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

def canonicalize_gen_trace(s: Any) -> str:
    """将 gen_trace 轻量归一化为字符串（可按需扩展）。"""
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    if NORMALIZE:
        # 统一换行符 & 去两端空白
        s = s.replace("\r\n", "\n").replace("\r", "\n").strip()
    return s

def digest_text(s: str) -> str:
    """对文本求 SHA-256 哈希，避免用超长字符串做键。"""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def to_target_schema(src: Dict[str, Any], is_bug_value: str) -> Dict[str, Any]:
    """映射到目标结构。"""
    return {
        "gen_trace": src.get("gen_trace"),
        "judgement": {
            "is_bug": is_bug_value,
            "reason": src.get("reason")
        },
        "reasoning": src.get("process")
    }


In [7]:
def coerce_bool(v) -> bool | None:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        if v == 1: return True
        if v == 0: return False
        return None
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ("true","t","yes","y","1"): return True
        if s in ("false","f","no","n","0"): return False
        return None
    return None

def parsed_is_bug_true(obj: Dict[str, Any]) -> bool:
    v = None
    p = obj.get("parsed")
    if isinstance(p, dict):
        v = p.get("is_bug")
    return coerce_bool(v) is True

def parsed_is_bug_false(obj: Dict[str, Any]) -> bool:
    v = None
    p = obj.get("parsed")
    if isinstance(p, dict):
        v = p.get("is_bug")
    return coerce_bool(v) is False

def build_output_item(src: Dict[str, Any], is_bug_value: str) -> Dict[str, Any]:
    """构建最终输出项：
    - 保留 gen_trace, thoughts
    - parsed -> judgement
    - 删 entry_id
    """
    out = {
        "gen_trace": src.get("gen_trace"),
        "judgement": src.get("parsed"),
        "thoughts": src.get("thoughts"),
    }
    # 强制设置 is_bug 以防原 parsed 没写明
    if isinstance(out["judgement"], dict):
        out["judgement"]["is_bug"] = is_bug_value
    return out


In [8]:
A_raw = load_json_list(A_PATH)
B_raw = load_json_list(B_PATH)
nA, nB = len(A_raw), len(B_raw)
print(f"加载完成：A={nA}, B={nB}")

N = min(nA, nB)
selected_yes, selected_no = [], []

for i in range(N):
    a = A_raw[i]
    b = B_raw[i]
    if parsed_is_bug_true(a) and parsed_is_bug_false(b):
        selected_yes.append(build_output_item(a, "yes"))
        selected_no.append(build_output_item(b, "no"))

print(f"筛选后可成对数量: {len(selected_yes)}（应等于 {len(selected_no)}）")


加载完成：A=1773, B=1773
筛选后可成对数量: 1164（应等于 1164）


In [10]:
pair_n = min(len(selected_yes), len(selected_no))
paired_sequence = []
for i in range(pair_n):
    paired_sequence.append(selected_yes[i])
    paired_sequence.append(selected_no[i])

paired_seq_path = os.path.join(OUTPUT_DIR, "trace_all_thinking_summary_by_gemini_sorted.json")
save_as_json(paired_seq_path, paired_sequence)

print(f"交替顺序输出已保存: {paired_seq_path} (总数: {len(paired_sequence)})")


交替顺序输出已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_all_thinking_summary_by_gemini_sorted.json (总数: 2328)


In [11]:
combined = []
for i in range(pair_n):
    combined.append(selected_yes[i])
    combined.append(selected_no[i])

random.Random(RANDOM_SEED).shuffle(combined)
all_shuf_path = os.path.join(OUTPUT_DIR, "trace_all_thinking_summary_by_gemini_shuffled.json")
save_as_json(all_shuf_path, combined)

print(f"打乱版输出已保存: {all_shuf_path} (总数: {len(combined)})")


打乱版输出已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_all_thinking_summary_by_gemini_shuffled.json (总数: 2328)


In [ ]:
# 加载
A_PATH = ""
C_PATH = ""

A_raw = load_json_list(A_PATH)   # 只含 gen_trace, reason
C_raw = load_json_list(C_PATH)

print(f"加载完成：A={len(A_raw)}, C={len(C_raw)}")

# 1) 找出 C 中 bug-free-trace == "" 的 gen_trace 哈希计数
missing_in_B_counter = Counter()
for item in C_raw:
    if item.get("bug-free-trace", "") == "":
        gt = canonicalize_gen_trace(item.get("gen_trace"))
        missing_in_B_counter[digest_text(gt)] += 1

print(f"C 中 bug-free-trace 为空的样本数: {sum(missing_in_B_counter.values())}")

# 2) 从 A 中删除对应样本，同时记录被删除的
A_filtered = []
A_removed  = []   # 👈 新增：存被删掉的样本
removed = 0

# 使用副本计数器以免原始 counter 被破坏
delete_quota = Counter(missing_in_B_counter)

for obj in A_raw:
    gt = canonicalize_gen_trace(obj.get("gen_trace"))
    h  = digest_text(gt)
    if delete_quota[h] > 0:
        delete_quota[h] -= 1
        A_removed.append(obj)   # ✅ 保存被删掉的
        removed += 1
    else:
        A_filtered.append(obj)

print(f"A 原始条数: {len(A_raw)}")
print(f"根据 C 删掉条数: {removed}")
print(f"A 过滤后条数: {len(A_filtered)}")

# 保存两个结果
A_filtered_path = os.path.join(OUTPUT_DIR, "trace_gen_buggy_balanced.json")
A_removed_path  = os.path.join(OUTPUT_DIR, "trace_gen_buggy_extra.json")

save_as_json(A_filtered_path, A_filtered)
save_as_json(A_removed_path, A_removed)

print(f"削减后的 A 已保存: {A_filtered_path}")
print(f"被删掉的 A 样本已保存: {A_removed_path} (数量: {len(A_removed)})")


加载完成：A=1777, C=1777
C 中 bug-free-trace 为空的样本数: 4
A 原始条数: 1777
根据 C 删掉条数: 4
A 过滤后条数: 1773
削减后的 A 已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_gen_buggy_balanced.json
被删掉的 A 样本已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_gen_buggy_extra.json (数量: 4)


In [21]:
# 加载
A_raw = load_json_list(A_PATH)
B_raw = load_json_list(B_PATH)
C_raw = load_json_list(C_PATH)

len_A0, len_B0, len_C0 = len(A_raw), len(B_raw), len(C_raw)
print(f"加载完成：A={len_A0}, B={len_B0}, C={len_C0}")

# 1) 在 C 中找 bug-free-trace == "" 的样本，收集其 gen_trace 的哈希计数
missing_in_B_counter = Counter()
miss_indices = []
for idx, item in enumerate(C_raw):
    if item.get("bug-free-trace", "") == "":
        gt = canonicalize_gen_trace(item.get("gen_trace"))
        missing_in_B_counter[digest_text(gt)] += 1
        miss_indices.append(idx)

print(f"C 中 bug-free-trace 为空的样本数: {sum(missing_in_B_counter.values())}")

# 2) 用“多重计数”逻辑从 A 中删掉对应数量的 gen_trace（按哈希匹配）
A_filtered = []
removed = 0
for obj in A_raw:
    gt = canonicalize_gen_trace(obj.get("gen_trace"))
    h = digest_text(gt)
    if missing_in_B_counter[h] > 0:
        missing_in_B_counter[h] -= 1
        removed += 1
        # 丢弃该样本（即“删除”）
    else:
        A_filtered.append(obj)

print(f"A 原始条数: {len_A0}")
print(f"根据 C 删掉条数: {removed}")
print(f"A 过滤后条数: {len(A_filtered)}")


加载完成：A=1777, B=1773, C=1777
C 中 bug-free-trace 为空的样本数: 4
A 原始条数: 1777
根据 C 删掉条数: 4
A 过滤后条数: 1773


In [22]:
# A_yes 和 B_no 已经准备好
len_yes, len_no = len(A_yes), len(B_no)
pair_n = min(len_yes, len_no)

# 按顺序交替拼接
paired_sequence = []
for i in range(pair_n):
    paired_sequence.append(A_yes[i])  # yes
    paired_sequence.append(B_no[i])   # no
    
paired_json_path = os.path.join(OUTPUT_DIR, "trace_all_sorted_reasoning_gpt5.json")
save_as_json(paired_json_path, paired_sequence)

print(f"交替顺序输出已保存: {paired_json_path}")
print(f"总数: {len(paired_sequence)} (yes={len_yes}, no={len_no}, 取最小长度 {pair_n})")


交替顺序输出已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_all_sorted_reasoning_gpt5.json
总数: 3546 (yes=1773, no=1773, 取最小长度 1773)


In [23]:
combined = A_yes + B_no
random.Random(RANDOM_SEED).shuffle(combined)

all_shuf_json_path = os.path.join(OUTPUT_DIR, "trace_all_shuffle_reasoning_gpt5.json")
save_as_json(all_shuf_json_path, combined)

print(f"混合打乱输出已保存: {all_shuf_json_path} (总数: {len(combined)})")


混合打乱输出已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_all_shuffle_reasoning_gpt5.json (总数: 3546)


In [ ]:
# —— 配置 D 路径（按需修改） ——
D_PATH = ""

def extract_is_bug(obj: dict):
    """
    从对象里提取 is_bug，兼容：
    - obj["judgement"]["is_bug"]
    - obj["is_bug"]
    返回小写 'yes'/'no' 或 None
    """
    v = None
    j = obj.get("judgement")
    if isinstance(j, dict) and "is_bug" in j:
        v = j["is_bug"]
    elif "is_bug" in obj:
        v = obj["is_bug"]
    if isinstance(v, str):
        v = v.strip().lower()
        if v in ("yes", "no"):
            return v
    return None

D_raw = load_json_list(D_PATH)
print(f"D 总条数: {len(D_raw)}")

from collections import Counter

D_yes_counter = Counter()
D_no_counter  = Counter()

for item in D_raw:
    is_bug = extract_is_bug(item)
    gt = canonicalize_gen_trace(item.get("gen_trace"))
    h = digest_text(gt)
    if is_bug == "yes":
        D_yes_counter[h] += 1
    elif is_bug == "no":
        D_no_counter[h]  += 1
    else:
        # 如缺少标签，默认计入总量但不区分；这里直接忽略，确保严格按标签筛选
        pass

print(f"D-yes 计数: {sum(D_yes_counter.values())}, D-no 计数: {sum(D_no_counter.values())}")


D 总条数: 2212
D-yes 计数: 1106, D-no 计数: 1106


In [16]:
# A_yes, B_no 已在前面构造（分别是映射后的目标结构）
# 但为了哈希匹配，需要访问原始 gen_trace；A_yes/B_no 里也保留了 gen_trace 字段

selected_yes = []
selected_no  = []

# 多重计数：遇到匹配就减一，直到该 hash 计数为 0
D_yes_counter_copy = Counter(D_yes_counter)
D_no_counter_copy  = Counter(D_no_counter)

for obj in A_yes:
    gt = canonicalize_gen_trace(obj.get("gen_trace"))
    h  = digest_text(gt)
    if D_yes_counter_copy[h] > 0:
        selected_yes.append(obj)
        D_yes_counter_copy[h] -= 1

for obj in B_no:
    gt = canonicalize_gen_trace(obj.get("gen_trace"))
    h  = digest_text(gt)
    if D_no_counter_copy[h] > 0:
        selected_no.append(obj)
        D_no_counter_copy[h] -= 1

print(f"从 A_yes 中筛到: {len(selected_yes)} 条；从 B_no 中筛到: {len(selected_no)} 条")
# 理想情况下：len(selected_yes) == len(selected_no) == 1106


从 A_yes 中筛到: 1105 条；从 B_no 中筛到: 1106 条


In [ ]:
pair_n = min(len(selected_yes), len(selected_no))

paired_sequence_2212 = []
for i in range(pair_n):
    paired_sequence_2212.append(selected_yes[i])  # yes
    paired_sequence_2212.append(selected_no[i])   # no

paired_seq_2212_path = os.path.join(OUTPUT_DIR, "trace_all_sorted_reasoning_gemini_api_2212.json")
save_as_json(paired_seq_2212_path, paired_sequence_2212)

print(f"交替排序（D 子集）已保存: {paired_seq_2212_path}")
print(f"总数: {len(paired_sequence_2212)} (应为 2212；当前 yes={len(selected_yes)}, no={len(selected_no)}, 取最小的 2*{pair_n})")


交替排序（D 子集）已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_all_sorted_reasoning_gemini_api_2212.json
总数: 2210 (应为 2212；当前 yes=1105, no=1106, 取最小的 2*1105)


In [18]:
combined_2212 = selected_yes[:pair_n] + selected_no[:pair_n]
random.Random(RANDOM_SEED).shuffle(combined_2212)

all_shuf_2212_path = os.path.join(OUTPUT_DIR, "trace_all_shuffle_reasoning_gemini_api_2212.json")
save_as_json(all_shuf_2212_path, combined_2212)

print(f"混合打乱（D 子集）已保存: {all_shuf_2212_path}")
print(f"总数: {len(combined_2212)} (应为 2212)")


混合打乱（D 子集）已保存: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_all_shuffle_reasoning_gemini_api_2212.json
总数: 2210 (应为 2212)
